# Pipeline

Se incluye:

1. **Clasificación** para predecir `compatible`
2. **Regresión** para predecir `relationship_longevity_months`

## 0. Imports y configuración

In [1]:
# =========================
# 0. IMPORTS Y CONFIGURACIÓN
# =========================

# Manejo de datos
import pandas as pd
import numpy as np

# Modelado y validación
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

# Modelos
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Métricas de clasificación
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Métricas de regresión
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Visualización opcional
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

## 1. Carga de datos

In [2]:
# =========================
# 1. CARGA DE DATOS
# =========================

df = pd.read_csv("cupid_algorithm_dataset.csv")

print("Shape del dataset:", df.shape)
df.head()

Shape del dataset: (100000, 30)


,pair_id,a_age,a_education,a_location,a_career_field,a_career_ambition,a_openness,a_extraversion,a_agreeableness,a_conscientiousness,...,b_extraversion,b_agreeableness,b_conscientiousness,b_chronotype,b_spontaneity,b_love_language,b_emotional_expressiveness,compatibility_score,compatible,relationship_longevity_months
0,1,46,3,Suburban,Healthcare,0.23,0.67,0.78,0.32,0.49,...,0.61,0.67,0.50,0.20,0.19,Quality Time,0.73,43.5,0,60
1,2,32,2,Suburban,Tech,0.58,0.78,0.70,0.51,0.71,...,0.31,0.20,0.57,0.45,0.56,Physical Touch,0.84,60.4,0,59
2,3,25,4,Rural,Marketing,0.59,0.33,0.87,0.64,0.82,...,0.30,0.49,0.43,0.84,0.74,Physical Touch,0.48,74.3,1,84
3,4,38,4,Suburban,Finance,0.54,0.34,0.28,0.72,0.81,...,0.35,0.46,0.21,0.80,0.35,Receiving Gifts,0.41,58.0,0,70
4,5,36,2,Rural,Entrepreneurship,0.56,0.35,0.62,0.27,0.73,...,0.66,0.45,0.43,0.86,0.36,Acts of Service,0.50,69.8,1,68


## 2. Inspección inicial

In [3]:
# =========================
# 2. INSPECCIÓN INICIAL
# =========================

# Información general del dataset
df.info()

# Valores nulos por columna
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Resumen estadístico de variables numéricas
print("\nResumen estadístico:")
display(df.describe())

# Distribución de la variable objetivo de clasificación
print("\nDistribución de 'compatible':")
print(df["compatible"].value_counts())
print(df["compatible"].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   pair_id                        100000 non-null  int64  
 1   a_age                          100000 non-null  int64  
 2   a_education                    100000 non-null  int64  
 3   a_location                     100000 non-null  object 
 4   a_career_field                 100000 non-null  object 
 5   a_career_ambition              100000 non-null  float64
 6   a_openness                     100000 non-null  float64
 7   a_extraversion                 100000 non-null  float64
 8   a_agreeableness                100000 non-null  float64
 9   a_conscientiousness            100000 non-null  float64
 10  a_chronotype                   100000 non-null  float64
 11  a_spontaneity                  100000 non-null  float64
 12  a_love_language                

,pair_id,a_age,a_education,a_career_ambition,a_openness,a_extraversion,a_agreeableness,a_conscientiousness,a_chronotype,a_spontaneity,...,b_openness,b_extraversion,b_agreeableness,b_conscientiousness,b_chronotype,b_spontaneity,b_emotional_expressiveness,compatibility_score,compatible,relationship_longevity_months
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,...,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,50000.500000,36.558140,2.951980,0.496463,0.499959,0.500131,0.501397,0.498643,0.499697,0.500012,...,0.499897,0.499177,0.500100,0.500063,0.500779,0.500324,0.500310,63.122273,0.428810,68.906250
std,28867.657797,10.951788,1.117733,0.174898,0.223607,0.224783,0.223860,0.223444,0.224779,0.226265,...,0.223567,0.223989,0.223253,0.223312,0.224214,0.226398,0.225954,10.705780,0.494909,21.334574
min,1.000000,18.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,16.500000,0.000000,0.000000
25%,25000.750000,27.000000,2.000000,0.370000,0.330000,0.330000,0.330000,0.320000,0.330000,0.330000,...,0.330000,0.330000,0.330000,0.330000,0.330000,0.320000,0.330000,55.900000,0.000000,54.000000
50%,50000.500000,37.000000,3.000000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,...,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,63.000000,0.000000,69.000000
75%,75000.250000,46.000000,4.000000,0.620000,0.670000,0.670000,0.680000,0.670000,0.670000,0.670000,...,0.670000,0.670000,0.670000,0.670000,0.680000,0.680000,0.670000,70.300000,1.000000,83.000000
max,100000.000000,55.000000,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,100.000000,1.000000,120.000000



Distribución de 'compatible':
compatible
0    57119
1    42881
Name: count, dtype: int64
compatible
0    0.57119
1    0.42881
Name: proportion, dtype: float64


## 3. Bloque 1 — Clasificación de `compatible`

En este bloque:
- separamos `X` e `y`,
- hacemos train/test split estratificado,
- seleccionamos variables con `mutual_info_classif`,
- ajustamos una **Logistic Regression** con GridSearch,
- y la comparamos contra **Random Forest**.


In [4]:
# =========================
# 3. CLASIFICACIÓN: PREDECIR 'compatible'
# =========================

# Separamos variables predictoras y target
X_clf = df.drop(columns=[
    "compatible",
    "compatibility_score",
    "relationship_longevity_months"
])
y_clf = df["compatible"]

# Convertimos variables categóricas a numéricas
# drop_first=True evita redundancia innecesaria
X_clf = pd.get_dummies(X_clf, drop_first=True)

# Split estratificado:
# importante para que la proporción de clases se mantenga en train y test
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

print("Train shape:", X_train_clf.shape)
print("Test shape:", X_test_clf.shape)

Train shape: (80000, 51)
Test shape: (20000, 51)


In [5]:
# =========================
# 3.1 FEATURE SELECTION PARA CLASIFICACIÓN
# =========================

# Mutual information adecuada para clasificación binaria
mi_clf = mutual_info_classif(
    X_train_clf,
    y_train_clf,
    random_state=RANDOM_STATE
)

mi_clf_series = pd.Series(mi_clf, index=X_train_clf.columns).sort_values(ascending=False)

print("Top 15 features por mutual information:")
display(mi_clf_series.head(15))

# Seleccionamos las top features en lugar de un umbral fijo (Anteriormente teníamos threshold_clf = 0.01, 
# pero ninguna variable superaba el umbral)
threshold_clf = 0.001
selected_features_clf = mi_clf_series[mi_clf_series > threshold_clf].index.tolist()

# Si no pasa ninguna feature el umbral, usamos las 20 mejores
if len(selected_features_clf) == 0:
    selected_features_clf = mi_clf_series.head(20).index.tolist()

print(f"Número de features seleccionadas para clasificación: {len(selected_features_clf)}")
print(selected_features_clf)

# Reducimos el dataset solo a las features elegidas
X_train_clf_sel = X_train_clf[selected_features_clf]
X_test_clf_sel = X_test_clf[selected_features_clf]

Top 15 features por mutual information:


a_location_Urban              0.006749
a_location_Suburban           0.005392
a_career_ambition             0.005387
b_location_Suburban           0.005007
a_education                   0.003526
b_education                   0.003500
b_career_ambition             0.003379
b_career_field_Tech           0.003331
b_spontaneity                 0.003174
b_career_field_Finance        0.003113
b_location_Urban              0.002664
b_career_field_Marketing      0.002620
a_spontaneity                 0.002612
b_emotional_expressiveness    0.002378
a_age                         0.002312
dtype: float64

Número de features seleccionadas para clasificación: 30
['a_location_Urban', 'a_location_Suburban', 'a_career_ambition', 'b_location_Suburban', 'a_education', 'b_education', 'b_career_ambition', 'b_career_field_Tech', 'b_spontaneity', 'b_career_field_Finance', 'b_location_Urban', 'b_career_field_Marketing', 'a_spontaneity', 'b_emotional_expressiveness', 'a_age', 'a_love_language_Words of Affirmation', 'b_love_language_Physical Touch', 'a_agreeableness', 'a_emotional_expressiveness', 'a_love_language_Quality Time', 'pair_id', 'a_career_field_Engineering', 'a_career_field_Marketing', 'b_love_language_Quality Time', 'b_openness', 'b_career_field_Entrepreneurship', 'a_career_field_Education', 'b_agreeableness', 'a_extraversion', 'a_openness']


In [6]:
# =========================
# 3.2 MODELO 1: LOGISTIC REGRESSION + GRIDSEARCH
# =========================

# Probamos distintos niveles de regularización y penalización
param_grid_clf = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],
    "class_weight": [None, "balanced"]
}

grid_clf = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=param_grid_clf,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_clf.fit(X_train_clf_sel, y_train_clf)

best_log_clf = grid_clf.best_estimator_

print("Mejores hiperparámetros Logistic Regression:")
print(grid_clf.best_params_)

Mejores hiperparámetros Logistic Regression:
{'C': 10, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'liblinear'}


In [7]:
# =========================
# 3.3 EVALUACIÓN DE LOGISTIC REGRESSION
# =========================

y_pred_log = best_log_clf.predict(X_test_clf_sel)
y_prob_log = best_log_clf.predict_proba(X_test_clf_sel)[:, 1]

print("===== LOGISTIC REGRESSION =====")
print(f"Accuracy: {accuracy_score(y_test_clf, y_pred_log):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test_clf, y_prob_log):.4f}")

print("\nClassification report:")
print(classification_report(y_test_clf, y_pred_log))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_clf, y_pred_log))

===== LOGISTIC REGRESSION =====
Accuracy: 0.5121
ROC-AUC: 0.5123

Classification report:
              precision    recall  f1-score   support

           0       0.58      0.52      0.55     11424
           1       0.44      0.51      0.47      8576

    accuracy                           0.51     20000
   macro avg       0.51      0.51      0.51     20000
weighted avg       0.52      0.51      0.51     20000


Confusion matrix:
[[5886 5538]
 [4220 4356]]


In [8]:
# =========================
# 3.4 MODELO 2: RANDOM FOREST CLASSIFIER
# =========================

rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_clf.fit(X_train_clf_sel, y_train_clf)

y_pred_rf_clf = rf_clf.predict(X_test_clf_sel)
y_prob_rf_clf = rf_clf.predict_proba(X_test_clf_sel)[:, 1]

print("===== RANDOM FOREST CLASSIFIER =====")
print(f"Accuracy: {accuracy_score(y_test_clf, y_pred_rf_clf):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test_clf, y_prob_rf_clf):.4f}")

print("\nClassification report:")
print(classification_report(y_test_clf, y_pred_rf_clf))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_clf, y_pred_rf_clf))

===== RANDOM FOREST CLASSIFIER =====
Accuracy: 0.6338
ROC-AUC: 0.6786

Classification report:
              precision    recall  f1-score   support

           0       0.64      0.83      0.72     11424
           1       0.62      0.38      0.47      8576

    accuracy                           0.63     20000
   macro avg       0.63      0.60      0.60     20000
weighted avg       0.63      0.63      0.61     20000


Confusion matrix:
[[9433 1991]
 [5333 3243]]


In [9]:
# =========================
# 3.5 CROSS-VALIDATION DE CLASIFICACIÓN
# =========================

# Validación cruzada con el mejor modelo lineal
cv_log_clf = cross_val_score(
    best_log_clf,
    X_clf[selected_features_clf],
    y_clf,
    cv=5,
    scoring="roc_auc"
)

# Validación cruzada con Random Forest
cv_rf_clf = cross_val_score(
    rf_clf,
    X_clf[selected_features_clf],
    y_clf,
    cv=5,
    scoring="roc_auc"
)

print("===== CROSS-VALIDATION CLASIFICACIÓN =====")
print(f"Logistic Regression ROC-AUC medio: {cv_log_clf.mean():.4f}")
print(f"Random Forest ROC-AUC medio: {cv_rf_clf.mean():.4f}")

===== CROSS-VALIDATION CLASIFICACIÓN =====
Logistic Regression ROC-AUC medio: 0.5143
Random Forest ROC-AUC medio: 0.6791


In [10]:
# =========================
# 3.6 INTERPRETABILIDAD EN CLASIFICACIÓN
# =========================

# Coeficientes de Logistic Regression:
# ayudan a ver qué variables empujan más hacia una clase u otra
coef_clf = pd.DataFrame({
    "feature": selected_features_clf,
    "coef": best_log_clf.coef_[0]
}).sort_values(by="coef", key=abs, ascending=False)

print("Top 15 coeficientes absolutos de Logistic Regression:")
display(coef_clf.head(15))

# Importancias de Random Forest
rf_importance_clf = pd.DataFrame({
    "feature": selected_features_clf,
    "importance": rf_clf.feature_importances_
}).sort_values(by="importance", ascending=False)

print("Top 15 importancias de Random Forest:")
display(rf_importance_clf.head(15))

Top 15 coeficientes absolutos de Logistic Regression:


,feature,coef
25,b_career_field_Entrepreneurship,0.119878
0,a_location_Urban,0.098716
10,b_location_Urban,0.092472
7,b_career_field_Tech,0.075969
28,a_extraversion,0.074390
1,a_location_Suburban,0.071864
24,b_openness,-0.059697
27,b_agreeableness,-0.058962
17,a_agreeableness,0.053579
13,b_emotional_expressiveness,-0.049735


Top 15 importancias de Random Forest:


,feature,importance
2,a_career_ambition,0.071099
6,b_career_ambition,0.070239
18,a_emotional_expressiveness,0.070157
13,b_emotional_expressiveness,0.069125
20,pair_id,0.067004
8,b_spontaneity,0.064732
12,a_spontaneity,0.064078
24,b_openness,0.062951
29,a_openness,0.062937
28,a_extraversion,0.061343


## 4. Bloque 2 — Regresión de `relationship_longevity_months`

En este bloque:
- cambiamos el target a la duración de la relación,
- evitamos leakage eliminando `compatible`,
- usamos `mutual_info_regression`,
- ajustamos **Ridge** y **Random Forest Regressor**,
- y evaluamos con MAE, RMSE y R².


In [11]:
# =========================
# 4. REGRESIÓN: PREDECIR 'relationship_longevity_months'
# =========================

# Evitamos leakage:
# quitamos tanto el target de regresión como 'compatible'
X_reg = df.drop(columns=[
    "relationship_longevity_months",
    "compatible",
    "compatibility_score"
])
y_reg = df["relationship_longevity_months"]

# Convertimos variables categóricas a numéricas
X_reg = pd.get_dummies(X_reg, drop_first=True)

# Transformación logarítmica:
# útil cuando la variable objetivo está muy sesgada
y_reg_log = np.log1p(y_reg)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg_log,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train_reg.shape)
print("Test shape:", X_test_reg.shape)

Train shape: (80000, 51)
Test shape: (20000, 51)


In [12]:
# =========================
# 4.1 FEATURE SELECTION PARA REGRESIÓN
# =========================

mi_reg = mutual_info_regression(X_train_reg, y_train_reg)

mi_reg_series = pd.Series(mi_reg, index=X_train_reg.columns).sort_values(ascending=False)

print("Top 15 features por mutual information (regresión):")
display(mi_reg_series.head(15))

threshold_reg = 0.001
selected_features_reg = mi_reg_series[mi_reg_series > threshold_reg].index.tolist()

if len(selected_features_reg) == 0:
    selected_features_reg = mi_reg_series.head(20).index.tolist()

print(f"Número de features seleccionadas para regresión: {len(selected_features_reg)}")

X_train_reg_sel = X_train_reg[selected_features_reg]
X_test_reg_sel = X_test_reg[selected_features_reg]

Top 15 features por mutual information (regresión):


b_age                              0.004403
a_career_field_Finance             0.004267
b_location_Suburban                0.004170
a_spontaneity                      0.003835
b_career_ambition                  0.003743
b_career_field_Healthcare          0.003212
a_career_field_Education           0.003090
b_career_field_Law                 0.003038
b_love_language_Quality Time       0.003005
a_age                              0.002993
b_career_field_Entrepreneurship    0.002643
a_conscientiousness                0.002638
a_career_field_Marketing           0.002499
b_spontaneity                      0.002110
a_career_field_Science             0.002020
dtype: float64

Número de features seleccionadas para regresión: 18


In [13]:
# =========================
# 4.2 MODELO 1: RIDGE + GRIDSEARCH
# =========================

param_grid_reg = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

grid_reg = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid_reg,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_reg.fit(X_train_reg_sel, y_train_reg)

best_ridge_reg = grid_reg.best_estimator_

print("Mejor alpha para Ridge:")
print(grid_reg.best_params_)

Mejor alpha para Ridge:
{'alpha': 100}


In [14]:
# =========================
# 4.3 MODELO 2: RANDOM FOREST REGRESSOR
# =========================

rf_reg = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_reg.fit(X_train_reg_sel, y_train_reg)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
# =========================
# 4.4 FUNCIÓN DE EVALUACIÓN PARA REGRESIÓN
# =========================

def evaluate_regression(model, X_test, y_test_log, name):
    # Predicción en escala log
    y_pred_log = model.predict(X_test)

    # Volvemos a la escala original (meses)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test_log)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"===== {name} =====")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2: {r2:.4f}")
    print()

evaluate_regression(best_ridge_reg, X_test_reg_sel, y_test_reg, "RIDGE")
evaluate_regression(rf_reg, X_test_reg_sel, y_test_reg, "RANDOM FOREST REGRESSOR")

===== RIDGE =====
MAE: 17.3180
RMSE: 21.6114
R2: -0.0411

===== RANDOM FOREST REGRESSOR =====
MAE: 17.7083
RMSE: 22.0973
R2: -0.0885



In [18]:
# =========================
# 4.5 CROSS-VALIDATION DE REGRESIÓN
# =========================

cv_ridge_reg = cross_val_score(
    best_ridge_reg,
    X_reg[selected_features_reg],
    y_reg_log,
    cv=5,
    scoring="neg_mean_absolute_error"
)

cv_rf_reg = cross_val_score(
    rf_reg,
    X_reg[selected_features_reg],
    y_reg_log,
    cv=5,
    scoring="neg_mean_absolute_error"
)

print("===== CROSS-VALIDATION REGRESIÓN =====")
print(f"Ridge MAE medio (escala log): {-cv_ridge_reg.mean():.4f}")
print(f"Random Forest MAE medio (escala log): {-cv_rf_reg.mean():.4f}")

===== CROSS-VALIDATION REGRESIÓN =====
Ridge MAE medio (escala log): 0.2716
Random Forest MAE medio (escala log): 0.2773


In [19]:
# =========================
# 4.6 INTERPRETABILIDAD EN REGRESIÓN
# =========================

# Coeficientes de Ridge
coef_reg = pd.DataFrame({
    "feature": selected_features_reg,
    "coef": best_ridge_reg.coef_
}).sort_values(by="coef", key=abs, ascending=False)

print("Top 15 coeficientes absolutos de Ridge:")
display(coef_reg.head(15))

# Importancias de Random Forest Regressor
rf_importance_reg = pd.DataFrame({
    "feature": selected_features_reg,
    "importance": rf_reg.feature_importances_
}).sort_values(by="importance", ascending=False)

print("Top 15 importancias de Random Forest Regressor:")
display(rf_importance_reg.head(15))

Top 15 coeficientes absolutos de Ridge:


,feature,coef
10,b_career_field_Entrepreneurship,0.015344
7,b_career_field_Law,-0.010598
14,a_career_field_Science,0.007383
15,b_career_field_Tech,0.007042
5,b_career_field_Healthcare,-0.004549
4,b_career_ambition,-0.002933
8,b_love_language_Quality Time,0.002760
6,a_career_field_Education,-0.002742
1,a_career_field_Finance,-0.002667
3,a_spontaneity,0.001772


Top 15 importancias de Random Forest Regressor:


,feature,importance
11,a_conscientiousness,0.154935
3,a_spontaneity,0.138835
13,b_spontaneity,0.137473
0,b_age,0.130100
9,a_age,0.130031
4,b_career_ambition,0.126081
2,b_location_Suburban,0.019950
8,b_love_language_Quality Time,0.018606
16,a_love_language_Words of Affirmation,0.017683
17,a_love_language_Quality Time,0.017339


## 5. Conclusiones

### Sobre clasificación (Clasificación de `compatible`)
Tras eliminar variables que introducían fuga de información, el modelo de clasificación presenta un rendimiento más realista, con un ROC-AUC de aproximadamente 0.68 utilizando Random Forest. Este resultado indica que el problema es inherentemente complejo y no puede resolverse mediante relaciones lineales simples, como demuestra el bajo rendimiento de Logistic Regression (ROC-AUC ≈ 0.51).

El análisis de importancia de variables revela que la compatibilidad entre individuos depende de múltiples factores, incluyendo rasgos de personalidad, ambición profesional, educación y ubicación, sin que exista una variable claramente dominante. Esto sugiere que la compatibilidad es un fenómeno multifactorial y no lineal.

Asimismo, se identificó la presencia de variables irrelevantes como pair_id, que deben ser eliminadas para evitar patrones espurios en el modelo.

En conjunto, el modelo Random Forest ofrece un rendimiento moderado, capturando parcialmente la complejidad del problema, aunque aún presenta dificultades para identificar correctamente la clase positiva (relaciones compatibles), lo que indica margen de mejora mediante técnicas más avanzadas o ingeniería de variables.

### Sobre regresión (Regresión de `relationship_longevity_months`)
El modelo de regresión presenta un rendimiento bajo, con valores de R² negativos tanto en Ridge como en Random Forest, lo que indica que los modelos son incapaces de explicar la variabilidad de la duración de las relaciones mejor que una predicción basada en la media.

El análisis de importancia de variables muestra que no existe ninguna característica con capacidad predictiva significativa, y que la información disponible en el dataset es insuficiente para modelar adecuadamente este fenómeno.

A diferencia del problema de clasificación, donde se identificaban patrones no lineales, en este caso los modelos más complejos no aportan mejoras, lo que sugiere una falta de señal en los datos.

Esto indica que la duración de una relación es un fenómeno altamente complejo y dependiente de variables no observadas en el dataset, por lo que sería necesario incorporar nuevas fuentes de información para mejorar el rendimiento del modelo.

## 6. Mejoras

1. Eliminar pair_id
2. Ajustar class imbalance:
class_weight='balanced'
3.Probar XGBoost / LightGBM